# Field Transformers Deep Dive

Field transformers operate on individual columns/fields within records, restructuring
nested data, splitting multi-valued fields, generating IDs, and normalizing values.

This notebook tests all 12 field transformer providers:

| Transformer | Purpose |
|-------------|--------|
| JSONFlattenerProvider | Extract nested JSON string into top-level fields |
| JSONArrayExpanderProvider | Expand JSON arrays into separate rows |
| JSONArrayFlattenerProvider | Flatten JSON arrays into delimited strings |
| JSONValueFlattenerProvider | Extract specific values from JSON objects |
| CommaFlattenerProvider | Join list values into comma-separated strings |
| CommaSplitProvider | Split comma-separated strings into lists |
| EmbeddedJSONFieldTransformer | Parse embedded JSON and merge into record |
| PairedFlattenerProvider | Flatten paired key-value structures |
| RegexCleanerProvider | Clean field values using regex patterns |
| EnumStandardizer | Map variant values to standard enum labels |
| UuidGeneratorTransformer | Generate unique IDs for records lacking them |
| TimestampNormalizer | Normalize date/time formats to ISO 8601 |

**Prerequisites:**
- `pip install graphrag-toolkit-document-graph` (base install)
- No Neptune or AWS credentials required


In [ ]:
from graphrag_toolkit.document_graph.transform.transformer_provider_config import TransformerProviderConfig
from graphrag_toolkit.document_graph.transform.field_transformers.json_flattener import JSONFlattenerProvider
from graphrag_toolkit.document_graph.transform.field_transformers.json_array_expander import JSONArrayExpanderProvider
from graphrag_toolkit.document_graph.transform.field_transformers.json_array_flattener import JSONArrayFlattenerProvider
from graphrag_toolkit.document_graph.transform.field_transformers.json_value_flattener import JSONValueFlattenerProvider
from graphrag_toolkit.document_graph.transform.field_transformers.comma_flattener import CommaFlattenerProvider
from graphrag_toolkit.document_graph.transform.field_transformers.comma_split_provider import CommaSplitProvider
from graphrag_toolkit.document_graph.transform.field_transformers.embedded_json import EmbeddedJSONFieldTransformer
from graphrag_toolkit.document_graph.transform.field_transformers.paired_flattener import PairedFlattenerProvider
from graphrag_toolkit.document_graph.transform.field_transformers.regex_cleaner_provider import RegexCleanerProvider
from graphrag_toolkit.document_graph.transform.field_transformers.standardize_enum import EnumStandardizer
from graphrag_toolkit.document_graph.transform.field_transformers.uuid_generator import UuidGeneratorTransformer
from graphrag_toolkit.document_graph.transform.field_transformers.normalize_timestamp import TimestampNormalizer

## JSON Flattener

Extract fields from a JSON string column into top-level record properties.
Useful for metadata columns stored as serialized JSON.


In [ ]:
records = [
    {'id': 'CTRL-001', 'control_data': '{"framework": "NIST", "category": "Access Control", "severity": "High"}'},
    {'id': 'CTRL-002', 'control_data': '{"framework": "SOC2", "category": "Monitoring", "severity": "Medium"}'}
]
config = TransformerProviderConfig(name='flatten_control', args={'field': 'control_data', 'prefix': 'ctrl_'})
transformer = JSONFlattenerProvider(config)
result = transformer.transform(records)
print("JSONFlattenerProvider:")
for r in result:
    print(r)

## JSON Array Expander

Expand a JSON array field into multiple rows — one row per array element.
Useful for one-to-many relationships encoded in a single column.


In [ ]:
records = [
    {'finding_id': 'F-101', 'evidence': '[{"type": "LOG", "source": "CloudTrail"}, {"type": "ALERT", "source": "GuardDuty"}]'}
]
config = TransformerProviderConfig(name='expand_evidence', args={
    'field': 'evidence',
    'skip_empty_arrays': True,
    'conflict_resolution': 'prefix',
    'auto_add_discovered': True
})
transformer = JSONArrayExpanderProvider(config)
result = transformer.transform(records)
print("JSONArrayExpanderProvider:")
print(f"Input: 1 record -> Output: {len(result)} records")
for r in result:
    print(r)

## JSON Array Flattener

Flatten a JSON array into a delimited string (e.g., `["a","b"]` → `"a,b"`).
Useful when you need a single string property, not multiple rows.


In [ ]:
records = [
    {'policy_id': 'POL-01', 'controls': '["AC-1", "AC-2", "AC-3"]'}
]
config = TransformerProviderConfig(name='flatten_controls', args={
    'field': 'controls',
    'prefix': 'control',
    'suffix': ' #00'
})
transformer = JSONArrayFlattenerProvider(config)
result = transformer.transform(records)
print("JSONArrayFlattenerProvider:")
for r in result:
    print(r)

## JSON Value Flattener

Extract specific values from JSON objects by key path.


In [ ]:
records = [
    {'user': 'admin', 'project_ids': 'PRJ-100, PRJ-200, PRJ-300', 'project_names': 'IAM Upgrade, SOC Setup, Audit Prep'}
]
config = TransformerProviderConfig(name='flatten_projects', args={
    'id_field': 'project_ids',
    'name_field': 'project_names',
    'prefix': 'project',
    'max_items': 5
})
transformer = JSONValueFlattenerProvider(config)
result = transformer.transform(records)
print("JSONValueFlattenerProvider:")
for r in result:
    print(r)

## Comma Flattener

Join list-typed values into comma-separated strings for storage as a single property.


In [ ]:
records = [
    {'user': 'analyst_01', 'permissions': 'read:logs, write:alerts, admin:dashboards'}
]
config = TransformerProviderConfig(name='flatten_permissions', args={
    'field': 'permissions',
    'prefix': 'perm',
    'suffix': ' #00',
    'max_items': 10
})
transformer = CommaFlattenerProvider(config)
result = transformer.transform(records)
print("CommaFlattenerProvider:")
for r in result:
    print(r)

## Comma Split

Split comma-separated strings into Python lists for downstream processing.


In [ ]:
records = [
    {'control_id': 'AC-2', 'assigned_teams': 'IAM Team, SOC, Platform Engineering'}
]
config = TransformerProviderConfig(name='split_teams', args={
    'fields': ['assigned_teams'],
    'separator': ',',
    'strip_whitespace': True
})
transformer = CommaSplitProvider(config)
result = transformer.transform(records)
print("CommaSplitProvider:")
print(f"Input: 1 record -> Output: {len(result)} records")
for r in result:
    print(r)

## Embedded JSON

Parse an embedded JSON string and merge its keys directly into the parent record.


In [ ]:
records = [
    {'alert_id': 'ALT-500', 'metadata': '{"source": "GuardDuty", "region": "us-east-1", "account_id": "123456789012"}'}
]
config = TransformerProviderConfig(name='extract_metadata', args={
    'field': 'metadata',
    'prefix': 'meta_'
})
transformer = EmbeddedJSONFieldTransformer(config)
result = transformer.transform(records)
print("EmbeddedJSONFieldTransformer:")
for r in result:
    print(r)

## Paired Flattener

Flatten paired key-value structures (e.g., `{"key": "region", "value": "us-east-1"}`) into direct properties.


In [ ]:
records = [
    {'user': 'admin', 'account_ids': 'ACC-01, ACC-02', 'account_names': 'Production, Staging'}
]
config = TransformerProviderConfig(name='pair_accounts', args={
    'id_field': 'account_ids',
    'name_field': 'account_names',
    'prefix': 'account',
    'suffix': '#00',
    'max_items': 5
})
transformer = PairedFlattenerProvider(config)
result = transformer.transform(records)
print("PairedFlattenerProvider:")
for r in result:
    print(r)

## Regex Cleaner

Clean field values using regex patterns — remove special characters, extract substrings, or normalize formats.


In [ ]:
records = [
    {'id': 'V-001', 'description': '<p>Critical <b>CVE-2024-1234</b> found in production.</p>'},
    {'id': 'V-002', 'description': '<div>Unpatched <i>OpenSSL</i> vulnerability.</div>'}
]
config = TransformerProviderConfig(name='strip_html', args={
    'patterns': ['<[^>]*>'],
    'replacement': '',
    'fields': ['description']
})
transformer = RegexCleanerProvider(config)
result = transformer.transform(records)
print("RegexCleanerProvider:")
for r in result:
    print(r)

## Enum Standardizer

Map variant values (typos, abbreviations, legacy codes) to a standard set of enum labels.


In [ ]:
records = [
    {'id': 1, 'severity': 'Critical'},
    {'id': 2, 'severity': 'HIGH'},
    {'id': 3, 'severity': 'med'},
    {'id': 4, 'severity': 'informational'}
]
config = TransformerProviderConfig(name='standardize_severity', args={
    'field': 'severity',
    'mapping': {
        'critical': 'CRITICAL',
        'high': 'HIGH',
        'med': 'MEDIUM',
        'medium': 'MEDIUM',
        'low': 'LOW',
        'informational': 'INFO'
    },
    'case_insensitive': True
})
transformer = EnumStandardizer(config)
result = transformer.transform(records)
print("EnumStandardizer:")
for r in result:
    print(r)

## UUID Generator

Generate unique identifiers for records that lack stable IDs — ensures every node has a distinct identity in the graph.


In [ ]:
records = [
    {'control': 'AC-1', 'title': 'Access Control Policy'},
    {'control': 'AC-2', 'title': 'Account Management'}
]
config = TransformerProviderConfig(name='gen_uuid', args={'target_field': 'node_id'})
transformer = UuidGeneratorTransformer(config)
result = transformer.transform(records)
print("UuidGeneratorTransformer:")
for r in result:
    print(r)

## Timestamp Normalizer

Normalize date/time values from various formats into consistent ISO 8601 strings for temporal queries.


In [ ]:
records = [
    {'event': 'login_attempt', 'timestamp': 'Jun 20, 2026 14:30:45'},
    {'event': 'privilege_escalation', 'timestamp': '2026/06/21 09:15 AM'},
    {'event': 'data_export', 'timestamp': '21-06-2026 23:59:59'}
]
config = TransformerProviderConfig(name='normalize_ts', args={
    'field': 'timestamp',
    'output_field': 'iso_timestamp'
})
transformer = TimestampNormalizer(config)
result = transformer.transform(records)
print("TimestampNormalizer:")
for r in result:
    print(r)

## Summary
All field transformers tested successfully:
- **JSONFlattenerProvider**: Flatten JSON objects into prefixed columns
- **JSONArrayExpanderProvider**: Expand JSON arrays into multiple records
- **JSONArrayFlattenerProvider**: Flatten JSON arrays into numbered columns
- **JSONValueFlattenerProvider**: Create JSON objects from paired fields
- **CommaFlattenerProvider**: Flatten comma-separated values into numbered columns
- **CommaSplitProvider**: Split comma-separated values into multiple records
- **EmbeddedJSONFieldTransformer**: Parse and flatten embedded JSON strings
- **PairedFlattenerProvider**: Flatten paired comma-separated fields
- **RegexCleanerProvider**: Clean text with regex patterns
- **EnumStandardizer**: Map values to standard enumerations
- **UuidGeneratorTransformer**: Generate UUIDs for records
- **TimestampNormalizer**: Normalize timestamps to ISO 8601